# ODI to Databricks Migration

## Target: `HR.TRG_EMP`

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook converts an ODI `INSERT` statement for `HR.TRG_EMP` from Oracle SQL to Databricks Spark SQL. It performs a full load from `HR.EMPLOYEES` into the target table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., F for Full, I for Incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "2. Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number")

## ETL Parameters

The following parameters control the ETL process. These are defined as Databricks widgets and can be set at runtime.

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
    ${ETL_PROC_WID} AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Merge into Target Table

**SCEN_TASK_NO {10}, {20}:** These tasks were empty or placeholder steps in the original ODI session.

**SCEN_TASK_NO {30}:** Insert records into `TRG_EMP` from `EMPLOYEES` table.

In [ ]:
%sql
INSERT INTO workspace.hr.trg_emp
(
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    HIRE_DATE,
    JOB_ID,
    SALARY,
    COMMISSION_PCT,
    MANAGER_ID,
    DEPARTMENT_ID
)
SELECT
    EMPLOYEES.EMPLOYEE_ID,
    EMPLOYEES.FIRST_NAME,
    EMPLOYEES.LAST_NAME,
    EMPLOYEES.EMAIL,
    EMPLOYEES.PHONE_NUMBER,
    EMPLOYEES.HIRE_DATE,
    EMPLOYEES.JOB_ID,
    EMPLOYEES.SALARY,
    EMPLOYEES.COMMISSION_PCT,
    EMPLOYEES.MANAGER_ID,
    EMPLOYEES.DEPARTMENT_ID
FROM
    workspace.hr.employees AS EMPLOYEES;

## Optimize Target Table

Optimize the target table to improve query performance. A generic ZORDER BY `employee_id` is applied.

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_emp ZORDER BY (employee_id);

## Cleanup

No temporary staging or flow tables were created in this process, so no specific cleanup is required here.

## Validation

Validate the results of the ETL process by checking record counts and sampling the target table.

In [ ]:
%sql
SELECT COUNT(*) AS final_record_count FROM workspace.hr.trg_emp;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_emp LIMIT 10;

## Conversion Notes and Manual Actions Required

*   **Data Types:** The column data types in `workspace.hr.trg_emp` are assumed to be compatible with `workspace.hr.employees`. If the target table `trg_emp` does not exist, its DDL must be created, inferring Spark SQL types from the Oracle `EMPLOYEES` table structure.
*   **Schema and Table Names:** `HR.TRG_EMP` and `HR.EMPLOYEES` were converted to `workspace.hr.trg_emp` and `workspace.hr.employees` respectively, following the Databricks naming conventions.
*   **Oracle Hints:** The Oracle hint `/*+ APPEND PARALLEL */` was removed as it is not applicable in Spark SQL/Delta Lake.
*   **SCEN_TASK_NO:** Empty `SCEN_TASK_NO` entries have been noted in markdown and are not associated with specific code cells.